
# PySpark Data Engineering: 30 Practical Q&A (Notebook-Style Markdown)

> **Purpose**: This notebook-style Markdown file consolidates core-to-advanced PySpark patterns you’ll use across Bronze/Silver/Gold layers, including DataFrame ops, windowed aggregations, joins, Delta Lake, and streaming. Each item includes a **concise solution** and the **reasoning behind it** so you can quickly recall the *why* not just the *how*.

---

## Part 1: Core DataFrame Operations & Data Cleaning
These tasks focus on reading, cleaning, and shaping raw data efficiently.

### 1) Read a CSV from S3 and infer schema
```python
# Assumes cluster has S3 access configured (IAM role/keys), and the path exists
# inferschema triggers a pass over the file to detect types

df = spark.read.csv("s3://bucket/path/data.csv", header=True, inferSchema=True)
```
**Explanation**: `inferSchema=True` detects column types from a sample/read pass; using `header=True` treats the first row as column names. This is simple but can be slower than specifying a schema—prefer explicit schemas in production for performance and type safety.

---

### 2) Drop rows where specific critical columns are null
```python
df_clean = df.dropna(subset=["transaction_id", "customer_id"])  # keep only rows where both are non-null
```
**Explanation**: `dropna(subset=...)` removes records missing critical business keys to ensure downstream joins and deduplication are reliable.

---

### 3) Fill null values with defaults across multiple columns
```python
df_filled = df.fillna({"status": "unknown", "amount": 0.0})
```
**Explanation**: `fillna` applies per-column default values, preventing null-propagation during computations (e.g., aggregates) and enabling safe filtering.

---

### 4) Rename multiple columns dynamically
```python
# Example: normalize spaces to underscores for SQL-friendliness
new_cols = [c.replace(" ", "_") for c in df.columns]
df_renamed = df.toDF(*new_cols)
```
**Explanation**: Renaming systematically enforces naming conventions that simplify downstream SQL queries and tooling compatibility.

---

### 5) Cast a string column to timestamp and filter last 30 days
```python
from pyspark.sql.functions import col, to_timestamp, current_date, date_sub

df_time = (
    df.withColumn("event_time", to_timestamp(col("time_str"), "yyyy-MM-dd HH:mm:ss"))
      .filter(col("event_time") >= date_sub(current_date(), 30))
)
```
**Explanation**: `to_timestamp` parses strings to `TimestampType`. Comparing with `current_date()` and `date_sub` is efficient and pushdown-friendly when possible.

---

### 6) Extract Year and Month from a date column
```python
from pyspark.sql.functions import year, month

df_ym = df.withColumn("year", year("event_date")).withColumn("month", month("event_date"))
```
**Explanation**: Adding calendar attributes supports partitioning, rollups, and dimensional modeling.

---

### 7) Flatten a nested Struct column
```python
# If `user_info` is a struct (e.g., {name: ..., age: ...})
df_flat = df.select("id", "user_info.*")
```
**Explanation**: Using `struct.*` expands nested fields into top-level columns, making them easier to query and transform.

---

## Part 2: Complex Transformations & Aggregations
Window functions and complex aggregations are staples of Silver/Gold processing.

### 8) Total, average, and max by category
```python
from pyspark.sql.functions import sum, avg, max

df_agg = df.groupBy("category").agg(
    sum("amount").alias("total_sales"),
    avg("amount").alias("avg_sales"),
    max("amount").alias("max_sale"),
)
```
**Explanation**: Grouped aggregations compute multiple metrics in a single shuffle, reducing passes over the data.

---

### 9) Efficiently count distinct in a massive dataset
```python
from pyspark.sql.functions import approx_count_distinct

# Much faster than exact countDistinct for large data
df_distinct = df.agg(approx_count_distinct("customer_id").alias("approx_unique_customers"))
```
**Explanation**: `approx_count_distinct` uses HyperLogLog++, providing near-accurate cardinality with sublinear memory and compute—ideal for big data.

---

### 10) Pivot rows to columns
```python
# Creates one row per store_id and a column per product_category, summing revenue

df_pivot = df.groupBy("store_id").pivot("product_category").sum("revenue")
```
**Explanation**: Pivoting is a shuffle-heavy operation; ensure the pivot column has bounded cardinality and consider `spark.sql.pivotMaxValues` limits.

---

### 11) Top 3 highest-paid employees per department
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
df_top = df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") <= 3)
```
**Explanation**: Window `row_number()` ranks rows within partitions; filtering by rank yields top-k per group without a post-aggregation join.

---

### 12) Running total of sales over time
```python
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

window_spec = (
    Window.partitionBy("store_id").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_running = df.withColumn("running_total", sum("sales").over(window_spec))
```
**Explanation**: Bounded/unbounded window frames enable cumulative metrics aligned to business ordering (e.g., by date).

---

### 13) 7-day moving average
```python
from pyspark.sql.functions import avg
from pyspark.sql.window import Window

window_spec = Window.partitionBy("stock_ticker").orderBy("date").rowsBetween(-6, 0)
df_moving_avg = df.withColumn("7_day_avg", avg("price").over(window_spec))
```
**Explanation**: A sliding window over the last 7 rows (not calendar days) computes a moving average. For calendar-based windows, consider time-range windowing on timestamp columns.

---

### 14) Explode an array column into rows
```python
from pyspark.sql.functions import explode, col

df_exploded = df.withColumn("item", explode(col("items_array")))
```
**Explanation**: `explode` normalizes arrays into 1:N row-level records for downstream joins and aggregations.

---

## Part 3: Joins & Optimization
Handling joins efficiently is critical for performance and cost.

### 15) Join two DataFrames and avoid duplicate column errors
```python
# If join key has the same name in both, passing the string uses it for equality join

df_joined = df1.join(df2, "customer_id", "inner")
```
**Explanation**: Supplying the key name avoids ambiguous column specs and generates a natural equality join on that column.

---

### 16) Records in A that are NOT in B
```python
df_missing = df_A.join(df_B, "id", "left_anti")
```
**Explanation**: `left_anti` returns rows from the left DataFrame with no matches on the right—perfect for anti-semi set operations.

---

### 17) Optimize a join between a massive table and a very small table
```python
from pyspark.sql.functions import broadcast

df_joined = large_df.join(broadcast(small_df), "key")
```
**Explanation**: Broadcast joins send the small table to all executors, avoiding a shuffle of the large table and drastically cutting join time.

---

### 18) Handle data skew during a join (salting)
```python
from pyspark.sql.functions import rand, explode, array, lit

# Add salt to the skewed key to spread hot keys across partitions
num_salts = 10

df_large_salted = large_df.withColumn("salt", (rand() * num_salts).cast("int"))
df_small_exploded = small_df.withColumn("salt", explode(array([lit(i) for i in range(num_salts)])))

df_joined = df_large_salted.join(df_small_exploded, ["skewed_key", "salt"])
```
**Explanation**: Skewed keys cause straggler tasks. Salting distributes hot keys over multiple partitions; after the join, you can re-aggregate to remove salt.

---

### 19) Coalesce multiple columns to the first non-null value
```python
from pyspark.sql.functions import coalesce, col

df_clean = df.withColumn("primary_contact", coalesce(col("mobile"), col("home_phone"), col("email")))
```
**Explanation**: `coalesce` returns the first non-null argument—useful for survivorship rules and canonicalization.

---

### 20) When and how to use a Python UDF
```python
# Prefer native functions for performance (Catalyst optimization & Tungsten codegen).
# Use UDFs only when logic cannot be expressed with built-in functions.
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def complex_logic(val):
    return val.upper() if val else "UNKNOWN"

my_udf = udf(complex_logic, StringType())
df_udf = df.withColumn("processed", my_udf(col("raw_data")))
```
**Explanation**: Python UDFs execute outside Spark’s JVM and incur (de)serialization overhead. Consider **SQL expressions**, **built-ins**, or **Pandas UDFs** (vectorized) before standard UDFs. In Spark 3+, **Scala/SQL** or **built-ins** often outperform.

---

### 21) Union two DataFrames with different schemas
```python
# Requires Spark 3.1+
df_union = df1.unionByName(df2, allowMissingColumns=True)
```
**Explanation**: `unionByName` aligns columns by name (not order) and fills missing columns with nulls—key for schema evolution.

---

## Part 4: Delta Lake & Architecture Concepts
Modern data platforms rely on Delta for ACID, schema enforcement, and time travel.

### 22) Write a DataFrame to Delta, partitioned by Year & Month
```python
(
    df.write
      .format("delta")
      .partitionBy("year", "month")
      .mode("append")
      .save("s3://bucket/silver/table")
)
```
**Explanation**: Partitioning improves pruning for time-series queries and reduces I/O. Choose partitions with bounded cardinality to avoid small-file explosion.

---

### 23) Generate a surrogate key for a Gold dimension table
```python
from pyspark.sql.functions import monotonically_increasing_id

# Note: monotonically_increasing_id is unique but not strictly sequential
# For sequential within a partition, use row_number over a deterministic ordering

df_sk = df.withColumn("dim_key", monotonically_increasing_id())
```
**Explanation**: Surrogate keys decouple dimension identity from source natural keys. For strict sequences, handle in a centralized key generator or with windowing + offset logic.

---

### 24) Upsert (MERGE) into a Delta table
```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "s3://bucket/silver/table")
(
    delta_table.alias("target").merge(
        df_updates.alias("source"),
        "target.id = source.id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
```
**Explanation**: Delta MERGE provides ACID-compliant upserts with automatic file and transaction management—essential for CDC, slowly changing dimensions, and idempotent loads.

---

### 25) Time Travel: query an older version of a Delta table
```python
df_v5 = (
    spark.read
         .format("delta")
         .option("versionAsOf", 5)
         .load("s3://bucket/silver/table")
)
```
**Explanation**: Time Travel enables point-in-time queries for reproducibility, auditing, and rollback testing by version or timestamp.

---

### 26) Optimize a Delta table and Z-Order
```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "s3://bucket/silver/table")
# Compacts small files and reorders data to cluster by frequently filtered columns
(
    delta_table.optimize().executeZOrderBy("customer_id")
)
```
**Explanation**: Compaction mitigates the small-files problem, improving read performance. Z-Ordering co-locates related data pages for faster predicate pruning on high-cardinality columns.

---

## Part 5: Performance Tuning & Streaming
Understanding the engine pays dividends in speed and cost.

### 27) Repartition vs. Coalesce
```python
# Repartition triggers a full shuffle; use to INCREASE partitions or fix skew

df_repart = df.repartition(200)

# Coalesce reduces partitions WITHOUT a shuffle; use to DECREASE partitions (e.g., before writes)

df_coal = df.coalesce(10)
```
**Explanation**: Choose partition counts to balance parallelism and overhead. Repartition for even data distribution; coalesce to reduce file count without the cost of a shuffle.

---

### 28) Persist a DataFrame to memory and disk
```python
from pyspark import StorageLevel

# cache() == persist(StorageLevel.MEMORY_AND_DISK) by default
# Use *_SER levels to store serialized data and reduce memory footprint

df.persist(StorageLevel.MEMORY_AND_DISK_SER)
```
**Explanation**: Persist reused DataFrames to avoid recomputation (lineage traversal). Pick storage levels based on memory pressure and reuse frequency.

---

### 29) Read streaming data from a landing-zone folder
```python
schema = "id INT, name STRING, value DOUBLE"

streaming_df = (
    spark.readStream
         .format("cloudFiles")        # Auto Loader (Databricks)
         .option("cloudFiles.format", "json")
         .schema(schema)
         .load("s3://bucket/landing_zone/")
)
```
**Explanation**: With Auto Loader (Databricks), `cloudFiles` incrementally and efficiently ingests new files with scalable listing. On open-source Spark, use `format("json")` + `readStream` and a known schema.

---

### 30) Write a streaming DataFrame to Delta with checkpointing
```python
query = (
    streaming_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", "s3://bucket/checkpoints/my_table")
        .start("s3://bucket/bronze/my_table")
)
```
**Explanation**: Checkpointing persists offsets and progress (WAL) for exactly-once/at-least-once semantics (depending on sink). Writing to Delta provides transactional guarantees for downstream readers.

---

## Bonus: Practical Notes & Gotchas
- **Schema management**: Prefer explicit schemas in production for speed and correctness; set `mergeSchema` only when necessary to handle evolution.
- **Partitioning strategy**: Keep partitions under a few thousand files; monitor small-file issues and compact periodically.
- **Skew & hotspots**: Detect via task timelines and stage skew metrics; treat with salting, AQE (adaptive skew join), or repartitioning.
- **UDF alternatives**: Try SQL expressions, built-ins, `expr`, **Pandas UDFs**, or **Scala UDFs** for better performance.
- **AQE**: Enable Adaptive Query Execution to handle skew joins, coalesce partitions post-shuffle, and optimize plans dynamically.

---

**End of Notebook Markdown**



# PySpark Data Engineering: 50 Interview-Focused Programming Q&A (Notebook-Style Markdown)

> **Scope**: Practical, commonly asked PySpark/Delta/Databricks/AWS data engineering tasks. Each item includes a **concise solution** and a brief **explanation** of the reasoning and trade-offs.

---

## Part 6: DataFrame Essentials & ETL Patterns

### 1) Read multiple CSVs with an explicit schema (wildcard)
```python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
])

df = (spark.read
          .schema(schema)
          .option("header", True)
          .csv("s3://bucket/data/2025/*/transactions-*.csv"))
```
**Explanation**: Explicit schemas are faster and safer than `inferSchema`. Wildcards let you batch-read partitioned files. Prefer schema-on-read for production-grade ETL.

---

### 2) Read JSON with multiline and capture corrupt records
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("user", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True),
])

df = (spark.read
          .schema(schema)
          .option("multiLine", True)
          .option("mode", "PERMISSIVE")
          .json("s3://bucket/raw/events/") )

bad = df.filter(df["_corrupt_record"].isNotNull())
```
**Explanation**: `PERMISSIVE` puts malformed rows into `_corrupt_record`. Use `multiLine=True` for pretty-printed JSON.

---

### 3) Enforce column order and rename using a mapping
```python
from pyspark.sql.functions import col

rename_map = {"Transaction ID": "transaction_id", "Customer ID": "customer_id", "Amt": "amount"}

df_renamed = df.select([col(k).alias(v) for k, v in rename_map.items()])
```
**Explanation**: Selecting with aliases both **renames** and **orders** columns deterministically.

---

### 4) Trim, lower-case, and strip special characters
```python
from pyspark.sql.functions import trim, lower, regexp_replace, col

df_clean = (df
    .withColumn("name", trim(lower(col("name"))))
    .withColumn("name", regexp_replace(col("name"), r"[^a-z0-9_ ]", ""))
)
```
**Explanation**: Text normalization reduces downstream duplicate keys and improves join matches.

---

### 5) Deduplicate by key keeping the latest record by timestamp
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

w = Window.partitionBy("id").orderBy(col("event_time").desc())

df_dedup = df.withColumn("rn", row_number().over(w)).filter("rn = 1").drop("rn")
```
**Explanation**: Classic dedup pattern: rank by recency per key and keep row_number == 1.

---

### 6) Normalize emails and phone numbers
```python
from pyspark.sql.functions import lower, regexp_replace

df_norm = (df
    .withColumn("email", lower(regexp_replace("email", r"\s+", "")))
    .withColumn("phone", regexp_replace("phone", r"[^0-9]", ""))
)
```
**Explanation**: Remove whitespace/symbols for consistent matching and validation.

---

### 7) Parse ISO8601 timestamps with timezones and convert to UTC
```python
from pyspark.sql.functions import to_timestamp, to_utc_timestamp, col

df_ts = (df
    .withColumn("ts_local", to_timestamp(col("ts_str")))
    .withColumn("ts_utc", to_utc_timestamp(col("ts_local"), col("timezone")))
)
```
**Explanation**: Parse first; then normalize to UTC using the provided timezone.

---

### 8) Split a delimited string and explode into rows
```python
from pyspark.sql.functions import split, explode, col

df_items = (df
    .withColumn("items", split(col("items_str"), ","))
    .withColumn("item", explode(col("items")))
    .drop("items")
)
```
**Explanation**: Normalize 1:N relationships from delimited fields for joins/analytics.

---

### 9) Pivot with a restricted set of values
```python
categories = ["A", "B", "C"]

df_p = df.groupBy("store").pivot("category", values=categories).sum("revenue")
```
**Explanation**: Restricting pivot values limits shuffle and avoids excessive columns.

---

### 10) Unpivot (melt) columns into rows using `stack`
```python
from pyspark.sql.functions import expr

unpivoted = df.select(
    "id",
    expr("stack(3, 'revenue', revenue, 'cost', cost, 'profit', profit) as (metric, value)")
)
```
**Explanation**: `stack(n, ...)` turns wide columns into key/value rows, a common SQL pattern in Spark.

---

## Part 7: Advanced Joins & Keys

### 11) Join on multiple conditions with null-safe equality
```python
from pyspark.sql.functions import col

joined = df1.join(df2, (col("id") == col("id2")) & (col("email").eqNullSafe(col("email2"))), "left")
```
**Explanation**: Use `eqNullSafe` (`<=>`) to treat null==null as true when that’s desired.

---

### 12) Implement SCD Type 2 (Delta MERGE)
```python
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

delta_tgt = DeltaTable.forPath(spark, "s3://bucket/gold/dim_customer")

# New rows have: natural_key, attributes..., effective_start, effective_end, is_current
# Close out matches and insert new current rows
(delta_tgt.alias("t").merge(df_updates.alias("s"), "t.natural_key = s.natural_key AND t.is_current = true")
 .whenMatchedUpdate(set={"effective_end": current_timestamp(), "is_current": "false"})
 .whenNotMatchedInsert(values={**{c: f"s.{c}" for c in df_updates.columns}, "effective_start": current_timestamp(), "effective_end": None, "is_current": "true"})
 .execute())
```
**Explanation**: Close the current record, then insert a new version. Ensure proper constraints/filters to avoid duplicates.

---

### 13) Self-join to find consecutive events (next event per user)
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import lead

w = Window.partitionBy("user").orderBy("event_time")
seq = df.withColumn("next_event", lead("event", 1).over(w))
```
**Explanation**: `lead/lag` provide intra-user sequencing without joins.

---

### 14) Fuzzy join on name similarity with Levenshtein distance
```python
from pyspark.sql.functions import levenshtein, col

cand = (dfA.crossJoin(dfB)
          .withColumn("dist", levenshtein(col("dfA_name"), col("dfB_name")))
          .filter(col("dist") <= 2))
```
**Explanation**: Works for small/filtered candidate sets; otherwise compute-blocking keys first to reduce the O(N^2) cross join.

---

### 15) Surrogate key as SHA-256 hash of multiple columns
```python
from pyspark.sql.functions import sha2, concat_ws

with_keys = df.withColumn("sur_key", sha2(concat_ws("||", "col1", "col2", "col3"), 256))
```
**Explanation**: Hash-based keys are deterministic, compact, and distribute well. Watch for collisions (rare with 256-bit).

---

### 16) Write/read with bucketing for efficient joins
```python
# Write bucketed table (Hive-managed)
(df
  .write
  .bucketBy(16, "customer_id")
  .sortBy("customer_id")
  .mode("overwrite")
  .saveAsTable("sales_bucketed"))

# Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # force bucketed sort-merge
```
**Explanation**: Bucketing co-locates keys to reduce shuffle on compatible joins. Best with Hive tables and consistent bucket counts/sorts.

---

### 17) Mitigate skew using AQE (Adaptive Query Execution)
```python
spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)
```
**Explanation**: AQE detects skew and splits skewed partitions at runtime to avoid stragglers.

---

### 18) Broadcast hint vs. function
```python
# Either API works
hinted = df_large.hint("broadcast").join(df_small, "key")
# or
from pyspark.sql.functions import broadcast
hinted2 = df_large.join(broadcast(df_small), "key")
```
**Explanation**: Hints influence the optimizer; broadcasting small dimension tables avoids shuffling the large fact.

---

### 19) Join and disambiguate overlapping columns with suffixes
```python
from pyspark.sql.functions import col

j = df1.alias("a").join(df2.alias("b"), col("a.id") == col("b.id"), "left")
result = j.select("a.id", *(col(f"a.{c}").alias(f"{c}_a") for c in ["col1","col2"]), *(col(f"b.{c}").alias(f"{c}_b") for c in ["col1","col2"]))
```
**Explanation**: Alias and explicit select prevent duplicate column name conflicts.

---

### 20) Filter rows with existence check (left-semi)
```python
semi = df_orders.join(df_valid_customers, "customer_id", "left_semi")
```
**Explanation**: `left_semi` returns only left rows that have matches, without bringing in right columns.

---

## Part 8: Aggregations & Windows

### 21) Build user sessions by gap (30 minutes)
```python
from pyspark.sql.functions import lag, when, sum as ssum, col, unix_timestamp
from pyspark.sql.window import Window

w = Window.partitionBy("user").orderBy("event_time")

gap = (unix_timestamp(col("event_time")) - unix_timestamp(lag("event_time").over(w)))

with_flags = df.withColumn("new_session", when((gap.isNull()) | (gap > 1800), 1).otherwise(0))

sessions = with_flags.withColumn("session_id", ssum("new_session").over(w))
```
**Explanation**: Mark new sessions when the gap exceeds 30 minutes; cumulative sum yields session groups.

---

### 22) Percentiles/quantiles per group
```python
from pyspark.sql.functions import expr

q = df.groupBy("category").agg(expr("percentile_approx(amount, 0.5, 10000)").alias("p50"))
```
**Explanation**: `percentile_approx` is scalable and accurate enough for large data.

---

### 23) Conditional aggregates
```python
from pyspark.sql.functions import sum, when, col

agg = df.groupBy("store").agg(
    sum("revenue").alias("total"),
    sum(when(col("channel") == "online", col("revenue")).otherwise(0)).alias("online"),
)
```
**Explanation**: Use `when/otherwise` inside aggregates to compute conditional sums in one pass.

---

### 24) Time-range window: rolling 1-hour sum
```python
from pyspark.sql.functions import sum as ssum
from pyspark.sql.window import Window

w = Window.partitionBy("sensor").orderBy("ts").rangeBetween(-3600, 0)  # if ts is long seconds

roll = df.withColumn("sum_1h", ssum("value").over(w))
```
**Explanation**: Use `rangeBetween` on a numeric/long timestamp. Convert timestamp to long seconds if needed.

---

### 25) Top-N per group with ties (dense_rank)
```python
from pyspark.sql.functions import dense_rank, col
from pyspark.sql.window import Window

w = Window.partitionBy("dept").orderBy(col("salary").desc())

top = df.withColumn("r", dense_rank().over(w)).filter("r <= 3")
```
**Explanation**: `dense_rank` keeps ties; `row_number` breaks them.

---

### 26) Median per group
```python
from pyspark.sql.functions import expr

median = df.groupBy("category").agg(expr("percentile_approx(value, 0.5)").alias("median"))
```
**Explanation**: True median is expensive; approximate percentile is standard in big data.

---

### 27) Find gaps and islands (consecutive days)
```python
from pyspark.sql.functions import lag, date_sub, sum as ssum, col
from pyspark.sql.window import Window

w = Window.partitionBy("id").orderBy("day")

with_break = df.withColumn("break", (lag("day").over(w) != date_sub(col("day"), 1)).cast("int"))

runs = with_break.withColumn("island_id", ssum("break").over(w))
```
**Explanation**: A new island starts when the previous day isn’t consecutive; cumulative sum gives run groups.

---

### 28) First/last non-null value per group
```python
from pyspark.sql.functions import first, last

fl = df.groupBy("id").agg(
    first("val", ignorenulls=True).alias("first_val"),
    last("val", ignorenulls=True).alias("last_val"),
)
```
**Explanation**: `ignorenulls=True` avoids nulls overshadowing actual values.

---

### 29) Inter-arrival time between events
```python
from pyspark.sql.functions import lag, col, unix_timestamp
from pyspark.sql.window import Window

w = Window.partitionBy("device").orderBy("ts")

with_gap = df.withColumn("prev_ts", lag("ts").over(w))             .withColumn("gap_seconds", unix_timestamp("ts") - unix_timestamp("prev_ts"))
```
**Explanation**: Useful for anomaly detection and throughput analysis.

---

### 30) Cumulative sum with reset on condition
```python
from pyspark.sql.functions import when, sum as ssum, col
from pyspark.sql.window import Window

w = Window.partitionBy("account").orderBy("date")

with_flag = df.withColumn("reset", when(col("txn_type") == "reset", 1).otherwise(0))

cum = with_flag.withColumn("run_group", ssum("reset").over(w))                .withColumn("running_balance", ssum("amount").over(Window.partitionBy("account", "run_group").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)))
```
**Explanation**: Create groups bounded by reset events, then compute cumulative sums within each group.

---

## Part 9: Files, Partitions & Performance

### 31) Overwrite partitions dynamically (Parquet)
```python
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
(
  df.write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet("s3://bucket/silver/sales")
)
```
**Explanation**: Dynamic overwrite replaces only touched partitions—safer/faster than full overwrite.

---

### 32) Compact small files by coalescing per partition
```python
# Choose target file count; avoid coalesce(1) globally for large data
compact = df.repartition("year", "month").coalesce(4)
(
  compact.write
         .mode("overwrite")
         .partitionBy("year", "month")
         .parquet("s3://bucket/silver/sales")
)
```
**Explanation**: Repartition by partition columns then coalesce to reduce files per partition.

---

### 33) Read only specific partitions (pruning)
```python
subset = (spark.read.parquet("s3://bucket/silver/sales")
              .filter("year = 2025 and month in (1,2,3)"))
```
**Explanation**: Predicate on partition columns enables partition pruning, saving I/O.

---

### 34) Control output file sizes
```python
spark.conf.set("spark.sql.files.maxRecordsPerFile", 2_000_000)

(df.repartition(200).write.mode("overwrite").parquet("s3://bucket/silver/facts"))
```
**Explanation**: Tune `maxRecordsPerFile` and partitions to balance file sizes and parallelism.

---

### 35) Delta schema evolution during write
```python
(df.write
  .format("delta")
  .mode("append")
  .option("mergeSchema", True)
  .save("s3://bucket/silver/delta_table"))
```
**Explanation**: `mergeSchema` lets new columns be added during append. Use judiciously and monitor table history.

---

### 36) Add/drop columns in Delta (SQL)
```sql
ALTER TABLE delta.`s3://bucket/silver/delta_table` ADD COLUMNS (new_col STRING);
ALTER TABLE delta.`s3://bucket/silver/delta_table` DROP COLUMN new_col;
```
**Explanation**: Delta supports DDL for schema changes; validate downstream dependencies before drops.

---

### 37) Vacuum a Delta table
```sql
VACUUM delta.`s3://bucket/silver/delta_table` RETAIN 168 HOURS;  -- 7 days
```
**Explanation**: Cleans old files not needed for time travel. Ensure retention settings comply with data governance.

---

### 38) Create a Bloom filter index (Databricks)
```sql
CREATE BLOOMFILTER INDEX bf_cust ON TABLE delta.`s3://bucket/gold/f_sales` FOR COLUMNS(customer_id) OPTIONS (fpp = 0.1, numItems = 100000000);
```
**Explanation**: Bloom filters speed up point-lookups on high-cardinality columns. **Databricks-only** feature.

---

## Part 10: Streaming Patterns

### 39) File streaming with explicit schema
```python
schema = "id STRING, ts TIMESTAMP, amount DOUBLE"

stream_df = (spark.readStream
                   .schema(schema)
                   .json("s3://bucket/landing/events/") )
```
**Explanation**: In streaming, schema inference is not supported; you must provide a schema.

---

### 40) Watermark + deduplicate on key and timestamp
```python
from pyspark.sql.functions import window

clean = (stream_df
    .withWatermark("event_time", "10 minutes")
    .dropDuplicates(["id", "event_time"]))
```
**Explanation**: Watermark bounds state and allows late data up to a threshold while preventing unbounded growth.

---

### 41) Stream-static join with watermark
```python
static_dim = spark.read.format("delta").load("s3://bucket/silver/dim_products")

joined = (stream_df
    .withWatermark("event_time", "15 minutes")
    .join(static_dim, "product_id"))
```
**Explanation**: Stream-static joins are common for enriching events; watermark controls state for stream side.

---

### 42) foreachBatch for idempotent upserts to Delta
```python
from delta.tables import DeltaTable

def upsert_to_delta(microbatch_df, batch_id):
    delta = DeltaTable.forPath(spark, "s3://bucket/bronze/events")
    (delta.alias("t").merge(microbatch_df.alias("s"), "t.id = s.id")
          .whenMatchedUpdateAll()
          .whenNotMatchedInsertAll()
          .execute())

query = (stream_df.writeStream
    .foreachBatch(upsert_to_delta)
    .option("checkpointLocation", "s3://bucket/checkpoints/events")
    .start())
```
**Explanation**: `foreachBatch` enables exactly-once upsert logic per micro-batch using Delta MERGE.

---

### 43) Trigger types: once / availableNow / processingTime
```python
# Run a single micro-batch and then stop
q1 = stream_df.writeStream.trigger(once=True).start("s3://bucket/bronze/out1")

# Process all available data quickly and stop (Databricks)
q2 = stream_df.writeStream.trigger(availableNow=True).start("s3://bucket/bronze/out2")

# Fixed interval micro-batches
q3 = stream_df.writeStream.trigger(processingTime="1 minute").start("s3://bucket/bronze/out3")
```
**Explanation**: Choose triggers that match latency and operational needs. `availableNow` is **Databricks-only**.

---

### 44) Auto Loader with schema evolution (Databricks)
```python
auto = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "s3://bucket/_schemas/events")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load("s3://bucket/landing/events"))
```
**Explanation**: Auto Loader scales file discovery and supports incremental schema evolution. **Databricks-only**.

---

### 45) Checkpointing and exactly-once sinks
```python
q = (stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "s3://bucket/checkpoints/sales")
    .start("s3://bucket/bronze/sales"))
```
**Explanation**: Checkpoints persist offsets and progress enabling end-to-end exactly-once with Delta sinks.

---

## Part 11: AWS & Databricks Integration

### 46) Configure S3 access (s3a) at runtime (example)
```python
hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.InstanceProfileCredentialsProvider")
# For dev/testing only (not recommended in prod):
# hconf.set("fs.s3a.access.key", "<ACCESS>")
# hconf.set("fs.s3a.secret.key", "<SECRET>")

df = spark.read.parquet("s3a://bucket/path/")
```
**Explanation**: Prefer IAM roles/instance profiles. Avoid embedding static keys in code.

---

### 47) Use AWS Glue Data Catalog as the metastore (concept)
```python
# When cluster is configured with Glue as the catalog, you can query directly:
spark.sql("SHOW DATABASES")
spark.sql("SHOW TABLES IN prod")
dim = spark.table("prod.dim_customers")
```
**Explanation**: With proper cluster configs (Spark/Hive metastore to Glue), Spark SQL seamlessly uses Glue as the catalog.

---

### 48) Read from Amazon Kinesis (requires connector)
```python
kinesis_df = (spark.readStream
    .format("kinesis")
    .option("streamName", "clickstream")
    .option("region", "ap-south-1")
    .option("initialPosition", "LATEST")
    .load())
```
**Explanation**: Requires the Kinesis connector on the cluster. Useful for real-time event ingestion on AWS.

---

### 49) Databricks widgets for parameterized notebooks
```python
# In a Databricks notebook cell
dbutils.widgets.text("p_date", "2026-03-10", "Process Date")
proc_date = dbutils.widgets.get("p_date")
```
**Explanation**: Widgets externalize parameters for scheduled jobs and ad-hoc runs.

---

### 50) Read secrets from Databricks Secret Scope
```python
endpoint = dbutils.secrets.get(scope="prod", key="pg_endpoint")
```
**Explanation**: Keep credentials out of notebooks and configs; reference via secret scopes.

---

## Bonus: Quick Tips
- Prefer **Parquet/Delta** for analytics; use **snappy** compression by default.
- Keep partition cardinality reasonable to avoid small-file problems; compact periodically.
- Turn on **AQE** and monitor **SQL UI** for skew and stage bottlenecks.
- For reliability, implement **idempotent writes** (e.g., deterministic partitioning + MERGE for CDC).

**End of 50 Q&A Notebook Markdown**
